# Big Data in Neuroimaging: Hands-on Exercises

**A visual tour of the things that get harder (not easier) when your dataset gets huge.**

This notebook uses a *simulated* brain dataset so that every exercise has a known "right answer" baked in. You don't need to be a programmer to follow along, for each exercise the code is already written and runs as-is. Your job is to **read the picture**, and in the *Your turn* cells, **change one highlighted value and re-run** to watch the picture change.

What we'll cover:

1. **Loading & inspecting** a large dataset
2. **Missingness**: why, in big data, it's usually *not* random
   - **2.5 Handling missingness**: imputation vs simply dropping the blanks
3. **Confounding**: how a relationship can be an illusion created by a third variable
4. **Prediction vs explanation**: why a "significant" model can still predict badly
5. **Effect size vs significance**: why almost everything becomes "significant" at scale
6. **Multiple comparisons**: why testing many things guarantees false alarms
7. **Site & scanner effects**: the multi-scanner harmonisation headache
8. **Analysis choices**: how defensible decisions change the result

---
> **Note.** The dataset is engineered so the teaching points reproduce reliably:
> - `symptoms` is driven by **age**, *not* by grey matter, so the GMV–symptoms link in Ex.3 is a pure age confound that vanishes once age is controlled.
> - `behaviour` has a **genuinely tiny** true correlation with `brain_measure` (true *r* ≈ 0.05), the engine for Ex.5.
> - Imaging variables go **missing more often** for older, sicker, higher-motion participants, the engine for Ex.2, and (because the data are simulated) the ground truth that lets Ex.2.5 grade imputation methods.
> - `thickness` carries a hidden **scanner/site** offset, a talking point in Ex.3 and the engine for Ex.7.
> - Ex.6 (multiple comparisons) uses freshly generated pure-noise "regions", so nothing is real by construction.

## Setup: run this first

This cell loads the tools and builds our simulated dataset. You only need to run it once.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# --- Make plots big and readable for a projector ---
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 100,
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
TEAL, ORANGE, GREY = "#1b9e9e", "#e6711b", "#9aa0a6"


def residualise(y, x):
    """Remove the linear effect of x from y (defined here so any exercise can use it)."""
    b = np.polyfit(x, y, 1)
    return y - (b[0] * x + b[1])


def make_brain_dataset(n=10000, seed=42, add_missingness=True):
    """Simulated neuroimaging dataset with deliberately engineered relationships."""
    rng = np.random.default_rng(seed)

    # Demographics (always observed)
    age = rng.uniform(45, 80, n)
    sex = rng.integers(0, 2, n)
    ses = rng.normal(0, 1, n)                       # socio-economic score (standardised)

    # Scanner/site -> shifts cortical thickness (a hidden confound)
    site = rng.integers(0, 3, n)
    site_shift = np.array([0.0, -0.06, 0.05])[site]

    # Head motion: worse with age; drives data quality & missingness
    motion = np.clip(0.5 + 0.02 * (age - 45) + rng.normal(0, 0.3, n), 0.1, None)

    # Total intracranial volume: depends on sex
    tiv = 1400 + 90 * sex + rng.normal(0, 60, n)

    # Grey matter volume: shrinks with age, scales with TIV
    gmv = 600 - 1.8 * (age - 45) + 0.18 * (tiv - 1400) + rng.normal(0, 20, n)

    # Cortical thickness: thins with age + scanner offset
    thickness = 2.7 - 0.004 * (age - 45) + site_shift + rng.normal(0, 0.08, n)

    # Clinical symptoms: rise with AGE (not caused by the brain measures!)
    symptoms = 10 + 0.30 * (age - 45) + 0.40 * motion - 0.8 * ses + rng.normal(0, 4, n)

    # Behaviour: a GENUINELY TINY true link to one brain measure (true r ~ 0.05)
    TRUE_R = 0.05
    z = rng.normal(0, 1, n)
    behaviour = TRUE_R * z + np.sqrt(1 - TRUE_R**2) * rng.normal(0, 1, n)
    brain_measure = z

    df = pd.DataFrame(dict(
        age=age, sex=sex, ses=ses, site=site, motion=motion,
        tiv=tiv, gmv=gmv, thickness=thickness,
        symptoms=symptoms, behaviour=behaviour, brain_measure=brain_measure,
    ))

    # Systematic missingness: imaging missing more often for older/sicker/higher-motion
    if add_missingness:
        lin = -2.5 + 0.045 * (df.age - 45) + 0.05 * (df.symptoms - 15) + 0.8 * (df.motion - 1)
        p_missing = 1 / (1 + np.exp(-lin))
        miss = rng.random(n) < p_missing
        for col in ["gmv", "thickness", "brain_measure"]:
            df.loc[miss, col] = np.nan
        df["imaging_missing"] = miss   # helper flag for the teaching plot in Ex.2

    return df


df = make_brain_dataset(n=10000, seed=42)
print(f"Dataset ready: {df.shape[0]:,} participants x {df.shape[1]} variables")

> **Why simulated, and what it stands in for.** We use fake data so every exercise has a *known* right answer to check against. But the structure mirrors the real big-data resources you'll actually meet:
> - **UK Biobank** (~500,000 participants, ~40,000 with brain MRI): demographics + imaging-derived phenotypes like TIV, GMV, cortical thickness.
> - **ABCD** (~12,000 children, multi-site, longitudinal).
> - **HCP** (deep imaging on a smaller sample).
> - **ENIGMA**: a consortium that meta-analyses results across dozens of sites worldwide (connects to the meta-analysis session).
>
> Every pitfall below, systematic missingness, confounding, tiny-but-significant effects, multiple comparisons, and scanner/site effects, is a *daily* reality in these datasets.

## Exercise 1: Loading & inspecting the data

Before any analysis, *look* at your data. With 10,000 rows you can't read it line by line, so you summarise and you plot. Run the cells below to (a) peek at the first rows, (b) get summary numbers, and (c) see the shape of every variable at a glance.

**Step 1: peek at the raw rows.** `df.head()` prints the first five participants. Even with 10,000 rows, this quick glance tells you what columns exist, what *type* each is (continuous numbers, 0/1 flags like `sex`, category codes like `site`), and whether anything is obviously broken: impossible ages, an all-zero column, stray text where numbers should be. Think of it as opening the spreadsheet and looking at the top.

In [ ]:
df.head()

**Step 2: summarise every column at once.** `df.describe()` reports, for each variable, the count, the mean, the spread (`std`), and the min / quartiles / max. Two things to hunt for here: (1) a **`count` below 10,000** is your first sign of missing data: notice `gmv`, `thickness`, and `brain_measure` have fewer rows than the rest; and (2) values that make no physical sense for that measure (a negative volume, an age outside the recruited range). This one command replaces scrolling through thousands of rows by hand.

In [ ]:
df.describe().round(2)

**Step 3: see the *shape* of each variable.** A single number like the mean hides a lot: two variables can share a mean yet look completely different. A histogram shows the whole distribution: how the values pile up. The grid below draws one for every variable at once. Watch for the different *kinds* of shape: smooth bell curves (TIV, GMV), a flat/uniform band (age, by design), a pair of bars (the 0/1 `sex` flag), and lopsided "skewed" tails (motion). The shape matters because it decides which analyses and plots are honest to use later.

In [ ]:
# A picture of every variable's distribution at a glance
cols = ["age", "sex", "ses", "motion", "tiv", "gmv", "thickness", "symptoms", "behaviour"]
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, c in zip(axes.ravel(), cols):
    ax.hist(df[c].dropna(), bins=30, color=TEAL, edgecolor="white")
    ax.set_title(c)
fig.suptitle("Exercise 1: the shape of every variable", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()


**What to notice.** Some variables are roughly bell-shaped (TIV, GMV), some are flat by design (age), some are counts (sex), and some are skewed (motion). Knowing each variable's shape tells you which analyses and plots will make sense later.

## Exercise 2: Missingness: it's usually *not* random

In small lab studies, a missing value is often just bad luck. In big datasets, **who is missing is rarely random**, the people with missing imaging tend to differ systematically (older, more symptomatic, more head motion). If you quietly drop them, your "sample" silently becomes healthier and younger than the population you care about.

Three views, from simplest to most important.

### First, three flavours of missingness

Statisticians (following Donald Rubin) sort missing data into three *mechanisms*. The label decides what you're allowed to do about it, so it's worth getting straight before we look at any graph. The whole distinction turns on one question: **what decides whether a value is missing?**

- **MCAR: Missing Completely At Random.** The chance of being missing depends on *nothing at all*, a pure coin flip. *Example:* a technician occasionally forgets to save a file, unrelated to the patient. Under MCAR the people with missing scans are just a random slice of everyone, so dropping them loses precision but introduces **no bias**.

- **MAR: Missing At Random.** The chance of being missing depends **only on things you did observe**. *Example:* older or higher-motion participants fail to get a usable scan, and age and motion are sitting right there in your table. It's *not* random, but because the drivers are recorded you can model and correct for it (imputation, weighting).

- **MNAR: Missing Not At Random.** The chance of being missing depends on the **missing value itself** (or something you never measured), even after accounting for what you did observe. *Example:* the people with the smallest grey matter volume are the ones who couldn't complete the scan, so the scan is missing *because of* the very number you failed to record. No method fully rescues this from the data alone.

**How the three graphs below work as a filter.** We can't observe the mechanism directly, so we rule the options out one at a time:
- View 1 tells us *how much* is missing, but nothing about mechanism.
- View 2 tells us the missingness is *structured*, not scattered, a hint there's a common cause.
- View 3 is the decisive test: it compares *who* is missing vs present, which lets us **eliminate MCAR**. What's left (MAR vs MNAR) can't be settled by the plots: we reason about it, and we'll do that at the end.

**View 1: how *much* is missing?** The simplest question first: for each variable, what fraction of values are blank? The bar chart below sorts variables from least to most missing. You'll see the demographics are complete while the imaging measures are not. But a single percentage tells you nothing about *who* is missing: that's the question the next two views tackle.

In [ ]:
# View 1: how much is missing, per variable?
miss_pct = df.drop(columns="imaging_missing").isna().mean().sort_values() * 100
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(miss_pct.index, miss_pct.values, color=ORANGE)
ax.set_xlabel("% missing")
ax.set_title("Exercise 2a: missingness by variable")
for i, v in enumerate(miss_pct.values):
    if v > 0:
        ax.text(v + 0.3, i, f"{v:.0f}%", va="center")
plt.show()


**What View 1 rules out: nothing (yet).** We now know the three imaging variables are missing for a sizeable chunk of people. But the *amount* of missingness is silent about the *mechanism*: a variable that is missing 20% of the time could be perfectly MCAR (a fifth of files randomly lost) or strongly MNAR (the sickest fifth). A percentage alone can't tell MCAR, MAR and MNAR apart. We need to look at the *pattern* and, above all, at *who* is missing.

**View 2: is the missingness *patterned*?** Here each row is one participant and each column a variable; a black mark means that value is missing. If blanks were purely random they'd be scattered like TV static. Instead, watch how the missing imaging cells line up **row-wise**: the same people tend to be missing *all three* imaging measures at once. That alignment is the tell-tale that something systematic, not chance, decides who gets imaged.

In [ ]:
# View 2: the missingness "matrix", each row is a participant, black = missing
sample = df.drop(columns="imaging_missing").sample(300, random_state=0)
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(sample.isna().values, aspect="auto", cmap="Greys", interpolation="none")
ax.set_xticks(range(sample.shape[1]))
ax.set_xticklabels(sample.columns, rotation=45, ha="right")
ax.set_ylabel("participants (300 sampled)")
ax.set_title("Exercise 2b: pattern of missing values (black = missing)")
plt.show()

**What View 2 adds: structure, but not yet a verdict.** The black cells don't fall one-by-one at random: they line up in whole rows, because when a scan session fails you lose *all three* imaging measures at once for that person. That tells us there's a single, per-person cause (a scan either yielded usable data or it didn't), rather than each number dropping independently. But structure alone still doesn't reveal *whether that cause is random*: a whole-row dropout could in principle still be a coin flip per person (MCAR). So we **still can't eliminate MCAR** here: it just sets up the decisive test. For the actual verdict we have to ask whether the people who dropped out *differ* from those who stayed.

**View 3: *who* is missing? (the view that matters).** This is the real test. We split people into two groups, "has imaging" vs "imaging missing", and overlay their distributions for age, symptoms, and head motion. If the two coloured curves sit on top of each other, missingness is harmless and you can drop those rows without a second thought. If they're shifted apart, then dropping the missing rows quietly changes *which population your study actually describes*. The printed means underneath put a number on any shift you see.

In [ ]:
# View 3 (the important one): ARE the people with missing imaging different?
have = df[~df.imaging_missing]
miss = df[df.imaging_missing]
compare = ["age", "symptoms", "motion"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, c in zip(axes, compare):
    ax.hist(have[c], bins=30, alpha=0.7, color=TEAL, label="has imaging", density=True)
    ax.hist(miss[c], bins=30, alpha=0.7, color=ORANGE, label="imaging MISSING", density=True)
    ax.set_title(c)
    ax.set_yticks([])
axes[0].legend()
fig.suptitle("Exercise 2c: missingness is SYSTEMATIC, not random", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

for c in compare:
    print(f"{c:9s}:  has imaging = {have[c].mean():6.2f}   |   missing = {miss[c].mean():6.2f}")


**View 3 eliminates MCAR.** In 2c the orange (missing) crowd is clearly shifted toward older age, more symptoms, and more motion, and the printed means confirm it. Under MCAR those two curves would sit on top of each other, because the missing people would be a random slice of everyone. They plainly don't, so **the data are not MCAR.** And notice *what* they differ on: age, symptoms, and motion, all variables we fully observed. Missingness that is predicted by observed variables is exactly the signature of **MAR**.

So "drop the missing rows" is not a harmless cleanup step: it's a *decision about who your study is about*. Do it here and your remaining sample silently becomes younger, healthier, and lower-motion than the population you set out to describe.

### What about MNAR? (the one the plots can't settle)

We've ruled out MCAR. That leaves **MAR** and **MNAR**, and here's the uncomfortable part: **no graph can tell them apart.** The test would be *"once I account for age, symptoms and motion, does whether the scan is missing still depend on what the scan value itself would have been?"*, but that value is exactly what's missing, so the data can't answer it. We have to *reason* about the mechanism.

- **The MAR story (our case):** scans go missing because of *age, symptoms, and motion*, all observed. Nothing about the missing brain value itself decides whether it's missing. In this simulated dataset that's true **by construction**: the missingness was generated purely from age/symptoms/motion and never looks at `gmv`, `thickness`, or `brain_measure`. So we are safely in MAR, and we can correct for it using those observed drivers.

- **The MNAR story we *can't* rule out in real life:** imagine instead that people with the **smallest grey matter volume** (the most atrophy) were the very ones too unwell to finish the scan. Now the scan is missing *because of* the grey matter value we failed to record: the missingness depends on the unseen number itself. From the plots this could look identical to our MAR case, yet no imputation or weighting based on the observed columns can fully fix it.

**The honest takeaway.** MCAR is (largely) testable, and we rejected it. MAR vs MNAR is a *judgement call* about the data-collection process, not something the numbers decide for you. In real neuroimaging, motion is observed, but it often also tracks the underlying pathology; if the part driving dropout isn't fully captured by your recorded variables, you can drift from MAR toward MNAR. Our simulation is deliberately clean MAR so the lesson stays unambiguous.

In [ ]:
# YOUR TURN: pick a different variable to compare between the two groups.
# Try "ses", "tiv", or "thickness" and re-run.
VARIABLE = "site"          #  <-- 👉 CHANGE THIS

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df.loc[~df.imaging_missing, VARIABLE].dropna(), bins=30, alpha=0.7,
        color=TEAL, label="has imaging", density=True)
ax.hist(df.loc[df.imaging_missing, VARIABLE].dropna(), bins=30, alpha=0.7,
        color=ORANGE, label="imaging missing", density=True)
ax.set_title(f"Does '{VARIABLE}' differ by missingness?")
ax.set_yticks([]); ax.legend()
plt.show()


## Exercise 2.5: So what do we *do* about it? Handling missingness

We've diagnosed the missingness as **MAR** (Ex.2). Now the practical question: how do we analyse data that has holes in it? There's a menu of options, and they are *not* equally good:

- **Listwise deletion (complete-case analysis).** Keep only fully-observed rows (`df.dropna(...)`). Simple, and *unbiased under MCAR*, but under **MAR (our case) it biases the sample**, because the people you drop differ systematically.
- **Mean / median imputation.** Fill each blank with the column average. Keeps your sample size, but it **shrinks the variance, distorts correlations, and does not remove the MAR bias**: it just hides it.
- **Regression / multiple imputation (MICE, `IterativeImputer`).** Predict each missing value from the *other* variables, crucially including the ones that *drive* the missingness. Under MAR this recovers unbiased estimates. *Multiple* imputation repeats this several times and pools the results (Rubin's rules) so the extra uncertainty is honestly reflected.

Here we have a luxury impossible with real data: because the dataset is **simulated**, we can uncover the values that were hidden and **grade each method against the truth**.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (unlocks IterativeImputer)
from sklearn.impute import SimpleImputer, IterativeImputer

# Because the dataset is SIMULATED we can regenerate the values that were hidden
# (same seed, missingness off) and use them as ground truth, impossible with real data!
df_true   = make_brain_dataset(n=10000, seed=42, add_missingness=False)
true_gmv  = df_true["gmv"].values
missing   = df["gmv"].isna().values
true_mean = true_gmv.mean()

# --- Strategy A: listwise deletion (analyse complete cases only) ---
mean_drop = df["gmv"].dropna().mean()

# --- Strategy B: mean imputation (fill blanks with the observed mean) ---
gmv_meanimp = SimpleImputer(strategy="mean").fit_transform(df[["gmv"]]).ravel()

# --- Strategy C: iterative (MICE-style) imputation using the observed drivers ---
predictors = ["age", "sex", "ses", "site", "motion", "tiv", "symptoms", "behaviour", "gmv"]
gmv_iter = IterativeImputer(random_state=0, max_iter=10).fit_transform(df[predictors])[:, -1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: does each method recover the TRUE cohort mean GMV?
names = ["listwise\ndeletion", "mean\nimputation", "iterative\nimputation"]
means = [mean_drop, gmv_meanimp.mean(), gmv_iter.mean()]
bars = ax1.bar(names, means, color=[GREY, ORANGE, TEAL])
ax1.axhline(true_mean, color="black", ls="--", lw=2, label=f"true mean = {true_mean:.1f}")
ax1.set_ylim(min(means + [true_mean]) - 1.2, max(means + [true_mean]) + 1.2)
ax1.set_ylabel("estimated mean GMV")
ax1.set_title("Ex.2.5a: only iterative imputation\nrecovers the true cohort mean")
ax1.legend()
for b, m in zip(bars, means):
    ax1.text(b.get_x() + b.get_width() / 2, m, f"{m - true_mean:+.2f}", ha="center", va="bottom")

# Right: for the MISSING people, how close is each imputed value to the hidden truth?
ax2.scatter(true_gmv[missing], gmv_meanimp[missing], s=10, alpha=0.3, color=ORANGE, label="mean imputation")
ax2.scatter(true_gmv[missing], gmv_iter[missing],    s=10, alpha=0.3, color=TEAL,   label="iterative imputation")
lo, hi = true_gmv[missing].min(), true_gmv[missing].max()
ax2.plot([lo, hi], [lo, hi], color="black", ls="--", lw=2, label="perfect")
ax2.set_xlabel("true (hidden) GMV"); ax2.set_ylabel("imputed GMV")
ax2.set_title("Ex.2.5b: iterative tracks each person;\nmean imputation is one flat guess")
ax2.legend()
fig.tight_layout(); plt.show()

print(f"TRUE cohort mean GMV     : {true_mean:6.2f}")
print(f"A) listwise deletion     : {mean_drop:6.2f}   (bias {mean_drop - true_mean:+.2f})")
print(f"B) mean imputation       : {gmv_meanimp.mean():6.2f}   (bias {gmv_meanimp.mean() - true_mean:+.2f})")
print(f"C) iterative imputation  : {gmv_iter.mean():6.2f}   (bias {gmv_iter.mean() - true_mean:+.2f})")

**The teaching point.** The true cohort mean GMV is ≈ 576.5. **Listwise deletion** and **mean imputation** both land about +2 units too high, because the people who went missing were older with *less* grey matter, so ignoring them (or papering over them with the observed average) inflates the estimate. Mean imputation keeps your sample size but buys you *nothing* on bias; the left bars for deletion and mean-fill are essentially identical. Only **iterative imputation**, which is allowed to use age/symptoms/motion, pulls the estimate back onto the truth.

The right panel shows *why*: mean imputation gives every missing person the same flat value (a horizontal stripe), whereas iterative imputation reconstructs each individual's value close to the diagonal. Two caveats for real work: (1) *single* imputation understates uncertainty: use *multiple* imputation and pool for honest confidence intervals; (2) for **prediction** (Ex.4) you must impute *inside* the train/test split and never let the target leak into the imputation model: imputing brain measures from age and then predicting age would be cheating.

In [ ]:
# YOUR TURN: which variables should the imputer be allowed to use?
# The blanks are driven by age, symptoms and motion (Ex.2). Watch what happens to the
# bias when the imputer CAN vs CANNOT see those drivers.
USE_DRIVERS = True        #  <-- 👉 CHANGE THIS to False

if USE_DRIVERS:
    preds = ["age", "sex", "ses", "site", "motion", "tiv", "symptoms", "behaviour", "gmv"]
else:
    preds = ["sex", "ses", "site", "tiv", "behaviour", "gmv"]   # no age / symptoms / motion

g = IterativeImputer(random_state=0, max_iter=10).fit_transform(df[preds])[:, -1]
bias = g.mean() - true_mean
print(f"Imputer can see the missingness drivers (age, symptoms, motion)? {USE_DRIVERS}")
print(f"Imputed cohort mean GMV = {g.mean():.2f}   (true = {true_mean:.2f},  bias = {bias:+.2f})")
print("->", "Bias essentially gone, the imputer could account for WHY data went missing."
      if abs(bias) < 0.5 else
      "Bias is back, blind to the reason for missingness, imputation can't fix MAR bias.")

> **A pragmatic note: why the rest of this workshop just drops the blanks.** From here on the exercises call `df.dropna(...)` and analyse complete cases. As we've just shown, in real research that would be the *wrong* default under MAR. We do it here deliberately, for two reasons:
> 1. **The dataset is artificially generated** so its engineered relationships, the age confound (Ex.3), the tiny true effect (Ex.5), the site offset (Ex.7), are still plainly present in the complete-case subsample. Dropping rows costs us some statistical power, but it does **not** erase the lessons those exercises are built to teach.
> 2. It keeps each exercise focused on its **one** teaching point, rather than layering imputation machinery on top of every analysis.
>
> So read the `dropna` in the later cells as a **conscious simplification for teaching, not an endorsement**. With your own data, come back to this exercise and impute.

## Exercise 3: Confounding: when a relationship is an illusion

Here we ask: *is grey matter volume related to clinical symptoms?* The honest answer in this dataset is **no, not directly**. But a third variable (**age**) affects both, so a relationship *appears* out of thin air. This is the single most important idea in observational big-data analysis.

Our plan: look at the naive scatter, then **stop and reason about the causal structure with a diagram (a DAG)** before we trust it, and finally *check and correct* for what the diagram flags.

> **Before running the next cell,** consider the scatterplot you're about to see in Ex.3a: *does grey matter really drive symptoms, yes or no?* We'll then sketch a causal diagram and test it in 3b and 3c.

**Plot 1: the naive picture.** We simply scatter grey matter volume against symptoms and fit a straight line through the cloud. The line slopes clearly and the *p*-value is tiny, so the textbook verdict would be "a real, highly significant relationship: less grey matter goes with more symptoms." Hold on to that first impression; the next two plots will put it to the test.

In [ ]:
d = df.dropna(subset=["gmv"]).copy()

# Plot 1: the naive scatter, looks like a real relationship!
r, p = stats.pearsonr(d.gmv, d.symptoms)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(d.gmv, d.symptoms, s=8, alpha=0.25, color=GREY)
b = np.polyfit(d.gmv, d.symptoms, 1)
xs = np.array([d.gmv.min(), d.gmv.max()])
ax.plot(xs, b[0]*xs + b[1], color=ORANGE, lw=3)
ax.set_xlabel("grey matter volume"); ax.set_ylabel("clinical symptoms")
ax.set_title(f"Ex.3a: looks like a clear relationship (r = {r:.2f}, p < 0.001)")
plt.show()


### A quick analysis: could a confounder be skewing this?

That scatter looks convincing, but before we believe it, let's ask *what could produce this pattern besides a genuine GMV→symptoms effect*. The disciplined way to do that is to sketch our causal assumptions as a **DAG** (a *directed acyclic graph*): each box is a variable, each arrow a *direct causal effect*. Run the next cell to draw the structure we actually believe for these data. Two features jump out:

- **A confounder (a "fork").** `age` causes *both* grey matter volume (older → less GMV) *and* symptoms (older → more symptoms). That opens a **back-door path** `GMV ← age → symptoms`, which carries a **non-causal** association, the dashed grey link, and very likely most of what the scatter above is actually showing. The fix is to *close* that path by adjusting for age; we'll confirm the culprit by colouring the scatter by age (3b) and then remove age's effect from both variables (3c).

- **A collider.** `age`, `symptoms`, and `head motion` all feed into whether the scan ends up **missing**, a common *effect*, which makes it a **collider**. The key statistical point: a collider is the mirror image of a confounder: you must **not** adjust for it. Conditioning on a collider *induces* a spurious association between its causes (here, between symptoms and motion) where none exists, exactly the opposite of the confounder case.

*(The graph also marks head motion as a **mediator** on `age → motion → symptoms`: a cause of the outcome but **not** of GMV, so, importantly, it is **not** a confounder of GMV–symptoms.)*

**The rule a DAG lets you read off:** *close back-door paths by adjusting for the fork (age); never adjust for a collider or a mediator*: the first would manufacture a fake association, the second would erase a real one.

In [ ]:
# The causal graph we assume for these data: a confounder (fork), a mediator, and a collider.
PURPLE = "#7b5cc4"

fig, ax = plt.subplots(figsize=(9, 6))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")

nodes  = {"age": (0.42, 0.93), "gmv": (0.10, 0.50), "symptoms": (0.62, 0.50),
          "motion": (0.90, 0.80), "missing": (0.62, 0.06)}
labels = {"age": "AGE", "gmv": "grey matter\nvolume", "symptoms": "clinical\nsymptoms",
          "motion": "head\nmotion", "missing": "imaging\nMISSING?"}
face   = {"age": ORANGE, "gmv": TEAL, "symptoms": TEAL, "motion": PURPLE, "missing": "white"}

def arrow(a, b, **kw):
    d = dict(arrowstyle="-|>", lw=2.2, color="black", shrinkA=24, shrinkB=24)
    d.update(kw); ax.annotate("", xy=nodes[b], xytext=nodes[a], arrowprops=d)

# confounder (fork) + mediator (age -> motion -> symptoms) + collider (-> missing)
arrow("age", "gmv"); arrow("age", "symptoms")
arrow("age", "motion"); arrow("motion", "symptoms")          # the mediator path
arrow("symptoms", "missing", color=GREY); arrow("motion", "missing", color=GREY)  # into the collider

# observed (spurious) GMV–symptoms association
ax.annotate("", xy=nodes["symptoms"], xytext=nodes["gmv"],
            arrowprops=dict(arrowstyle="-", lw=1.8, color=GREY, ls="--", shrinkA=30, shrinkB=30))
ax.text(0.36, 0.44, "observed (spurious)", ha="center", color=GREY, fontsize=9, style="italic")

# role tags
ax.text(0.42, 0.99, "confounder (fork)", ha="center", color=ORANGE, fontsize=9, fontweight="bold")
ax.text(0.985, 0.80, "mediator", ha="left", va="center", color=PURPLE, fontsize=9, fontweight="bold")
ax.text(0.62, -0.02, "collider", ha="center", color=GREY, fontsize=9, fontweight="bold")

for k, (x, y) in nodes.items():
    shape = "square,pad=0.4" if k == "missing" else "round,pad=0.45"
    ax.text(x, y, labels[k], ha="center", va="center", fontsize=11, fontweight="bold",
            bbox=dict(boxstyle=shape, fc=face[k], ec=(GREY if k == "missing" else "black"),
                      alpha=0.92, ls=("--" if k == "missing" else "-")))

ax.set_title("Ex.3: the causal graph we assume, confounder, mediator, collider",
             fontsize=14, fontweight="bold")
plt.show()

**Plot 2: colour in the suspect.** Exactly the same points, but now each dot is tinted by that participant's **age** (dark = younger, bright = older). A structure appears that the plain black-and-white scatter hid: the low-GMV / high-symptom corner is crowded with *older* people, and the high-GMV / low-symptom corner with *younger* ones. Age is quietly pushing on *both* axes at once, and that is precisely what a confounder looks like.

In [ ]:
# Plot 2: the SAME scatter, now coloured by age, the illusion is revealed
fig, ax = plt.subplots(figsize=(8.5, 5.5))
sc = ax.scatter(d.gmv, d.symptoms, c=d.age, s=10, alpha=0.6, cmap="viridis")
ax.set_xlabel("grey matter volume"); ax.set_ylabel("clinical symptoms")
ax.set_title("Ex.3b: colour by AGE, older people have less GMV AND more symptoms")
plt.colorbar(sc, label="age (years)")
plt.show()


**Plot 3: take age out of both variables.** To ask whether *anything* links GMV and symptoms once age is accounted for, we statistically strip age's linear effect from each variable, this is called **residualising**, and then re-plot what's left over. If a genuine GMV–symptom link existed underneath, the tilt would survive. Watch instead as the line flattens to essentially horizontal: once age is removed, the "relationship" is gone.

In [ ]:
# Plot 3: remove age from BOTH variables, then look again -> the link is gone
def residualise(y, x):
    b = np.polyfit(x, y, 1)
    return y - (b[0]*x + b[1])

gmv_adj = residualise(d.gmv.values, d.age.values)
sym_adj = residualise(d.symptoms.values, d.age.values)
r_adj, p_adj = stats.pearsonr(gmv_adj, sym_adj)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(gmv_adj, sym_adj, s=8, alpha=0.25, color=TEAL)
b = np.polyfit(gmv_adj, sym_adj, 1)
xs = np.array([gmv_adj.min(), gmv_adj.max()])
ax.plot(xs, b[0]*xs + b[1], color=ORANGE, lw=3)
ax.set_xlabel("grey matter volume (age removed)")
ax.set_ylabel("symptoms (age removed)")
ax.set_title(f"Ex.3c: after controlling for age, r = {r_adj:.2f}, the relationship vanishes")
plt.show()

print(f"Before controlling for age:  r = {r:+.2f}")
print(f"After  controlling for age:  r = {r_adj:+.2f}   (essentially zero)")


**The teaching point.** The original "finding" (Ex.3a) was entirely an age effect. A residual plot (Ex.3c) is one honest way to ask: *is there anything left once the obvious confounder is removed?* Here, almost nothing.

> **Footnote for later.** Even the tiny leftover r ≈ -0.02 comes out "significant" because n is huge, which is exactly the trap of Exercise 5.

In [ ]:
# YOUR TURN: which variable is doing the confounding?
# Set this to "age", "tiv", "motion", or "none" and re-run.
# Only "age" should make the relationship collapse.
CONTROL_FOR = "age"        #  <-- 👉 CHANGE THIS

if CONTROL_FOR == "none":
    gx, gy = d.gmv.values, d.symptoms.values
    label = "nothing removed"
else:
    gx = residualise(d.gmv.values, d[CONTROL_FOR].values)
    gy = residualise(d.symptoms.values, d[CONTROL_FOR].values)
    label = f"{CONTROL_FOR} removed"

rr, pp = stats.pearsonr(gx, gy)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(gx, gy, s=8, alpha=0.25, color=TEAL)
ax.set_title(f"Controlling for '{CONTROL_FOR}'  ->  r = {rr:.2f}  ({label})")
ax.set_xlabel("grey matter volume"); ax.set_ylabel("symptoms")
plt.show()


## Exercise 4: Prediction vs explanation

A model can be statistically "significant" and still be **useless for predicting an individual**. We'll predict two things from the same brain + demographic variables:

- **age**: the brain predicts this fairly well (a tight diagonal),
- **symptoms**: the brain predicts this poorly (a shapeless blob),

even though *both* models are "significant." Big data does not automatically buy you clinically useful prediction.

> **Before running the cell,** which do you think the brain will predict better, someone's *age* or their *symptom score*?

**How the test works.** We split the participants into a **training** set, which the model learns from, and a separate **test** set it has never seen. We then ask the trained model to predict each test person and plot *predicted vs actual*. Points landing on the dashed diagonal are perfect predictions: a tight, narrow diagonal means good prediction, while a shapeless cloud means the model can't tell individuals apart. Crucially we run the *identical* procedure twice, once to predict age, once to predict symptoms, so any difference in the pictures is about the signal in the data, not the method.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy import stats

d4 = df.dropna(subset=["gmv", "thickness"]).copy()
features = ["gmv", "thickness", "tiv", "sex", "motion"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, target in zip(axes, ["age", "symptoms"]):
    X, y = d4[features].values, d4[target].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
    model = LinearRegression().fit(Xtr, ytr)
    pred = model.predict(Xte)
    r2 = r2_score(yte, pred)

    # Omnibus F-test: are the 5 predictors JOINTLY significant? (H0: all slopes = 0)
    n, k = Xtr.shape
    r2_in = model.score(Xtr, ytr)                      # in-sample R^2 for the F-test
    F = (r2_in / k) / ((1 - r2_in) / (n - k - 1))
    p = stats.f.sf(F, k, n - k - 1)

    ax.scatter(yte, pred, s=8, alpha=0.2, color=TEAL)
    lo, hi = yte.min(), yte.max()
    ax.plot([lo, hi], [lo, hi], color=ORANGE, lw=2, ls="--")
    ax.set_xlabel(f"actual {target}"); ax.set_ylabel(f"predicted {target}")
    ax.set_title(f"Predicting {target}:  test R² = {r2:.2f}")

    print(f"{target:9s}:  test R² = {r2:5.2f}   |   model F-test p = {p:.2e}   "
          f"({'SIGNIFICANT' if p < 0.05 else 'n.s.'})")

fig.suptitle("Exercise 4: same data, very different prediction quality",
             fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

print("\nBoth models are hugely 'significant' (p ≈ 0), with n≈8,000 even the weak")
print("symptoms model passes. Significance ≠ quality: read the R², not just the p-value.")


**The teaching point.** Both models would pass a significance test. But look at the *pictures*: predicted age hugs the dashed line, predicted symptoms is a cloud. "Significant" answers *is there signal?*; the scatter answers *can I trust a prediction for one person?*, and those are different questions.

In [ ]:
# YOUR TURN: try predicting a different target, or change the features.
TARGET = "age"                       #  <-- 👉 CHANGE THIS ("age","symptoms","behaviour")
FEATURES = ["gmv", "thickness", "tiv"]     #  <-- 👉 or change these

dd = df.dropna(subset=["gmv", "thickness", "brain_measure"]).copy()
X, y = dd[FEATURES].values, dd[TARGET].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
pred = LinearRegression().fit(Xtr, ytr).predict(Xte)
r2 = r2_score(yte, pred)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(yte, pred, s=8, alpha=0.2, color=TEAL)
lo, hi = yte.min(), yte.max()
ax.plot([lo, hi], [lo, hi], color=ORANGE, lw=2, ls="--")
ax.set_title(f"Predicting '{TARGET}'  ->  test R² = {r2:.2f}")
ax.set_xlabel(f"actual {TARGET}"); ax.set_ylabel(f"predicted {TARGET}")
plt.show()

# Reading the score: R² = 1 is perfect; R² = 0 is "no better than always guessing the average".
# A slightly NEGATIVE R² is not a bug, it means the model does WORSE than that flat guess,
# i.e. there is essentially no usable signal here. That is itself a valid, honest result.
if r2 < 0.05:
    print(f"R² = {r2:.2f}: this brain measure barely predicts '{TARGET}' at all, no usable signal.")
else:
    print(f"R² = {r2:.2f}: some predictive signal, but check the scatter to judge if it's trustworthy.")

## Exercise 5: Effect size vs statistical significance ⭐

This is the headline lesson. There's a **real but tiny** relationship in the data (true *r* ≈ 0.05, it explains about a quarter of one percent of the variance). Watch what happens to the *p*-value as the sample size grows, while the effect size itself **does not budge**.

The rule you learned in your methods course, "look for *p* < 0.05", quietly stops working at scale.

> 🔁 **In the *Your turn* cell below, re-run it several times at `n = 50`** and watch the result flicker between "significant" and "not significant". Then set `n = 50000` and re-run a few times: now it locks in as "significant" *every* time, even though the effect is the same tiny sliver.

**How we make this reliable.** A single sample could be a lucky fluke, so instead of drawing once we take **300 random subsamples** at each sample size (50, 500, 5,000, and 50,000 people). In each subsample we measure the effect size *r* and record whether it came out "significant" (*p* < 0.05). The top panel then shows the *average* effect size at each sample size, and the bottom panel shows the *percentage* of subsamples that were significant. The whole lesson lives in the gap between those two panels: one barely moves while the other shoots up.

In [ ]:
# The reliable version: for each sample size, draw many subsamples and ask
#   (a) what is the average effect size r?   (b) what fraction come out significant?
big = make_brain_dataset(n=120000, seed=7, add_missingness=False)
sizes = [50, 500, 5000, 50000]
rng = np.random.default_rng(0)

mean_r, pct_sig = [], []
for n in sizes:
    rs, sig = [], 0
    for _ in range(300):
        idx = rng.integers(0, len(big), n)
        r, p = stats.pearsonr(big.brain_measure.values[idx], big.behaviour.values[idx])
        rs.append(r); sig += (p < 0.05)
    mean_r.append(np.mean(rs)); pct_sig.append(100 * sig / 300)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
x = range(len(sizes))

ax1.bar(x, mean_r, color=TEAL)
ax1.axhline(0.05, color=GREY, ls="--", label="true effect (r = 0.05)")
ax1.set_ylabel("effect size (r)"); ax1.set_ylim(0, 0.12)
ax1.set_title("Effect size stays TINY and flat...")
ax1.legend()

ax2.bar(x, pct_sig, color=ORANGE)
ax2.set_ylabel("% of samples that are 'significant'")
ax2.set_title("...but 'significance' climbs to 100%")
ax2.set_xticks(list(x)); ax2.set_xticklabels([f"n = {n:,}" for n in sizes])
ax2.set_ylim(0, 105)
for i, v in enumerate(pct_sig):
    ax2.text(i, v + 2, f"{v:.0f}%", ha="center")

fig.suptitle("Exercise 5: bigger n manufactures significance, not importance",
             fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()


**The teaching point.** Top panel: the real effect is the same tiny sliver at every sample size. Bottom panel: at n = 50 almost nothing is significant; at n = 50,000 *everything* is. So in big data, "significant" can no longer mean "important": you're forced to ask the harder question: *is the effect big enough to matter?*

In [ ]:
# YOUR TURN: pick a single sample size and see one concrete result.
# Re-run a few times at n = 50 and watch it jump around;
# then try n = 50000 and watch it lock in as "significant" every time.
SAMPLE_SIZE = 5000          #  <-- 👉 CHANGE THIS (try 50, 500, 5000, 50000)

s = big.sample(SAMPLE_SIZE)
r, p = stats.pearsonr(s.brain_measure, s.behaviour)
verdict = "SIGNIFICANT (p < 0.05)" if p < 0.05 else "not significant"

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(s.brain_measure, s.behaviour, s=18, alpha=0.5, color=TEAL)
ax.set_xlabel("brain measure"); ax.set_ylabel("behaviour")
ax.set_title(f"n = {SAMPLE_SIZE:,}   ->   r = {r:.3f},  p = {p:.4f}\n{verdict}")
plt.show()

print(f"Effect size r = {r:+.3f}  (the TRUTH is ~0.05, tiny either way)")
print(f"p-value      = {p:.4f}  ->  {verdict}")


## Exercise 6: Multiple comparisons: test enough things and something "wins"

In big data you rarely test a single hypothesis: you test *hundreds* of brain regions against *dozens* of behaviours. Here's the catch: even when **nothing** is real, about **1 in 20** tests clears *p* < 0.05 by pure chance. That's what *p* < 0.05 *means*.

Below we correlate 100 **pure-noise** "brain regions" with an outcome. None of them has any true relationship. Count the false alarms, then see how a correction removes them.

**Reading the plot below.** Each bar is one of the 100 noise "regions", and its height is `-log10(p)`, a common way to draw *p*-values where **taller means a smaller *p*-value, i.e. more "significant"**. Bars poking above the grey dashed line clear the usual *p* < 0.05; the black dotted line is the stricter **Bonferroni** threshold, which divides 0.05 by the number of tests. Keep firmly in mind that we generated every region from pure noise, so *any* bar above a line is, by construction, a false alarm.

In [ ]:
rng = np.random.default_rng(1)
n_regions = 100
n_people = 500

outcome = rng.normal(0, 1, n_people)
# 100 "brain regions" that are PURE NOISE, none is truly related to the outcome
regions = rng.normal(0, 1, (n_people, n_regions))

pvals = np.array([stats.pearsonr(regions[:, j], outcome)[1] for j in range(n_regions)])

alpha = 0.05
n_uncorrected = int((pvals < alpha).sum())
n_bonferroni = int((pvals < alpha / n_regions).sum())   # Bonferroni: divide threshold by #tests

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = np.where(pvals < alpha, ORANGE, TEAL)
ax.bar(range(n_regions), -np.log10(pvals), color=colors)
ax.axhline(-np.log10(alpha), color=GREY, ls="--", label="p = 0.05 (uncorrected)")
ax.axhline(-np.log10(alpha / n_regions), color="black", ls=":", label="p = 0.05 (Bonferroni)")
ax.set_xlabel("brain region  (all 100 are pure noise)")
ax.set_ylabel("-log10(p)   (taller = more 'significant')")
ax.set_title(f"Ex.6: {n_uncorrected} 'significant' hits from PURE NOISE, {n_bonferroni} survive correction")
ax.legend()
plt.show()

print(f"Tested {n_regions} regions, NONE truly related to the outcome.")
print(f"  Uncorrected (p < 0.05):   {n_uncorrected} false 'discoveries'")
print(f"  After Bonferroni:         {n_bonferroni} survive")

**The teaching point.** Every orange bar is a *false alarm*, a "finding" from data with no signal at all. Report those uncorrected and you've published noise. This is why neuroimaging insists on **correction for multiple comparisons** (Bonferroni, FDR, cluster-based methods), and why **independent replication** and **meta-analysis** (the next session 👀) are the real test of whether an effect is genuine.

In [ ]:
# YOUR TURN: how many things are you testing? Change this and re-run a few times.
# The more tests you run, the more pure-noise "hits" you get before correction.
N_REGIONS = 100        #  <-- 👉 CHANGE THIS (try 20, 100, 1000)

rng2 = np.random.default_rng()          # no fixed seed -> re-run to see it vary
outcome = rng2.normal(0, 1, 500)
regions = rng2.normal(0, 1, (500, N_REGIONS))
pvals = np.array([stats.pearsonr(regions[:, j], outcome)[1] for j in range(N_REGIONS)])

n_hits = int((pvals < 0.05).sum())
n_survive = int((pvals < 0.05 / N_REGIONS).sum())
print(f"Testing {N_REGIONS} pure-noise regions:")
print(f"  {n_hits} came out 'significant' (expected ~{0.05*N_REGIONS:.0f} just by chance)")
print(f"  {n_survive} survive Bonferroni correction")

## Exercise 7: Site & scanner effects: the multi-site headache

Big datasets are pooled across many scanners and sites (that's how you *get* to half a million people). But different scanners nudge measurements up or down for purely **technical** reasons: coil, field strength, sequence, software version. Our `thickness` variable carries a hidden offset for each of 3 sites.

If site happens to line up with something you care about (say, one clinic scans the older patients), that technical offset **masquerades as biology**. Below: spot the offset, then remove it.

**Reading the two panels.** On the left we box-plot cortical thickness separately for each of the three sites. The people at each site are, on average, comparable, so if the boxes sit at different heights, that gap is the *scanner talking*, not the brain. On the right we apply the simplest possible fix: shift each site up or down so their averages line up, then re-plot. Watch the boxes drop into agreement. (The printed site means below quantify the offsets you're seeing.)

In [ ]:
d7 = df.dropna(subset=["thickness"]).copy()
sites = sorted(d7.site.unique())
site_labels = [f"site {s}" for s in sites]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

# Left: raw thickness differs by site, for purely technical reasons
raw = [d7.loc[d7.site == s, "thickness"] for s in sites]
axes[0].boxplot(raw)
axes[0].set_xticks(range(1, len(sites) + 1)); axes[0].set_xticklabels(site_labels)
axes[0].set_ylabel("cortical thickness")
axes[0].set_title("Ex.7a: same measure, three scanners\n(the gaps are technical, not brain)")

# Right: remove each site's mean offset, then re-plot -> the scanners line up
d7["thickness_adj"] = (
    d7.thickness - d7.groupby("site").thickness.transform("mean") + d7.thickness.mean()
)
adj = [d7.loc[d7.site == s, "thickness_adj"] for s in sites]
axes[1].boxplot(adj)
axes[1].set_xticks(range(1, len(sites) + 1)); axes[1].set_xticklabels(site_labels)
axes[1].set_title("Ex.7b: after removing each site's offset\n(harmonised, now comparable)")

fig.suptitle("Exercise 7: scanner offsets can look like real group differences",
             fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

for s in sites:
    print(f"site {s}: mean thickness = {d7.loc[d7.site == s, 'thickness'].mean():.3f}")

**The teaching point.** Subtracting each site's mean is the simplest possible harmonisation; real tools (e.g. **ComBat**) are fancier and also model the *spread*, not just the average. But the idea is the same: **measure the technical offset and remove it before you trust a group difference.** The danger case is when site is tangled with your variable of interest: then you literally cannot tell scanner from biology, and ignoring it can invent *or* hide an effect.

## Exercise 8: Analysis choices, how defensible decisions change the result

Same data, same question, two perfectly *defensible* analyst choices: opposite conclusions. This is how honest researchers reach significance without ever cheating, and it's why **pre-registration** matters. Run it and watch two reasonable people disagree.

In [ ]:
# Question: is grey matter related to symptoms?
# Analyst A controls for age. Analyst B doesn't. Both are "reasonable".
dA = df.dropna(subset=["gmv"]).copy()

# Analyst B: raw association
rB, pB = stats.pearsonr(dA.gmv, dA.symptoms)

# Analyst A: control for age first
gx = residualise(dA.gmv.values, dA.age.values)
gy = residualise(dA.symptoms.values, dA.age.values)
rA, pA = stats.pearsonr(gx, gy)

print("Same dataset, same question, two defensible choices:\n")
print(f"  Analyst B (no age control):  r = {rB:+.2f},  p = {pB:.1e}  ->  'GMV strongly linked to symptoms!'")
print(f"  Analyst A (controls age):    r = {rA:+.2f},  p = {pA:.2f}   ->  'No meaningful link.'")
print("\nNeither cheated. The analysis choices made the finding.")


## Wrap-up: what to take away

| Exercise | The trap | The habit that beats it |
|---|---|---|
| 1 · Loading & inspecting | You can't eyeball a million rows | Summarise *and* plot every variable first |
| 2 · Missingness | Dropping missing rows silently changes who your study is about | Always check *who* is missing |
| 2.5 · Handling it | Deletion and mean-fill leave the MAR bias in place | Multiple imputation using the missingness drivers |
| 3 · Confounding | A third variable invents a relationship | Plot it, colour by suspects, residualise |
| 4 · Prediction | "Significant" ≠ "predicts individuals" | Look at predicted-vs-actual |
| 5 · Effect size | Big n makes everything significant | Report and judge the *effect size* |
| 6 · Multiple comparisons | Test enough things and noise "wins" | Correct for the number of tests; replicate |
| 7 · Site effects | Scanners add technical offsets that look like biology | Harmonise across sites before comparing |
| 8 · Analysis choices | Defensible choices manufacture findings | Pre-register your analysis |

**One sentence:** in big data, statistical significance becomes cheap, so your real job shifts from *"is it significant?"* to *"is it real, is it big enough to matter, and am I sure it isn't a confound, a false alarm, or a scanner artefact?"*